In [7]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import LabelEncoder
import warnings 
warnings.filterwarnings("ignore")
from sklearn.metrics import classification_report,root_mean_squared_error, mean_squared_error

In [8]:
df = pd.read_csv("Titanic-Dataset.csv")
df

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
...,...,...,...,...,...,...,...,...,...,...,...,...
886,887,0,2,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.0000,NaN,S
887,888,1,1,"Graham, Miss. Margaret Edith",female,19.0,0,0,112053,30.0000,B42,S
888,889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,NaN,1,2,W./C. 6607,23.4500,NaN,S
889,890,1,1,"Behr, Mr. Karl Howell",male,26.0,0,0,111369,30.0000,C148,C


In [9]:
df.drop(columns = ["PassengerId","Name","Ticket","Fare","Cabin"], inplace = True)
df["Age"].fillna(value= 28, inplace =True)
df["Embarked"].fillna(value = "S", inplace = True)

le = LabelEncoder()
df["Sex"] = le.fit_transform(df["Sex"])
df["Embarked"] = le.fit_transform(df["Embarked"])

q1 = df["Age"].quantile(0.25)
q3 = df["Age"].quantile(0.75)
iqr = q3 - q1
lf = q1 - (1.5*iqr)
uf = q3 + (1.5*iqr)
df["Age"] = np.where(df["Age"] > uf, uf, df["Age"])
df["Age"] = np.where(df["Age"] < lf, lf, df["Age"])

y = df["Survived"]
x = df.drop(columns=["Survived"])

x_train,x_test,y_train,y_test = train_test_split(x, y, test_size = 0.3, random_state = 88)

In [10]:
lr = LogisticRegression()
dt = DecisionTreeClassifier(min_samples_split = 50, max_depth = 16, random_state = 88)
knn = KNeighborsClassifier()

In [11]:
lr.fit(x_train,y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


In [12]:
root_mean_squared_error(y_test,lr.predict(x_test))

0.47708693159476445

In [13]:
print("train accuracy : ",lr.score(x_train,y_train))
print("train accuracy : ",lr.score(x_test,y_test))

train accuracy :  0.8009630818619583
train accuracy :  0.7723880597014925


In [14]:
print(classification_report(y_train,lr.predict(x_train)))
print("*****************************************")
print(classification_report(y_test,lr.predict(x_test)))

              precision    recall  f1-score   support

           0       0.83      0.85      0.84       374
           1       0.76      0.73      0.75       249

    accuracy                           0.80       623
   macro avg       0.79      0.79      0.79       623
weighted avg       0.80      0.80      0.80       623

*****************************************
              precision    recall  f1-score   support

           0       0.84      0.80      0.82       175
           1       0.66      0.72      0.69        93

    accuracy                           0.77       268
   macro avg       0.75      0.76      0.75       268
weighted avg       0.78      0.77      0.77       268



In [15]:
dt.fit(x_train,y_train)

,criterion,'gini'
,splitter,'best'
,max_depth,16
,min_samples_split,50
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,88
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,class_weight,None


In [16]:
root_mean_squared_error(y_test,dt.predict(x_test))

0.47316022340738384

In [17]:
print("train accuracy : ",dt.score(x_train,y_train))
print("train accuracy : ",dt.score(x_test,y_test))

train accuracy :  0.826645264847512
train accuracy :  0.7761194029850746


### Grid Search For best parameters

In [18]:
para = {
    "n_estimators" : [50,100,200],
    "criterion" : ["gini","entropy"],
    "min_samples_split" : [25,50,75,100,150],
    "max_depth" : [2,4,6,8,10]
}
rf = RandomForestClassifier(random_state = 88)
gs = GridSearchCV(rf, param_grid = para, verbose = 1, n_jobs = -1)
gs.fit(x_train,y_train)

Fitting 5 folds for each of 150 candidates, totalling 750 fits


,estimator,RandomForestC...ndom_state=88)
,param_grid,"{'criterion': ['gini', 'entropy'], 'max_depth': [2, 4, ...], 'min_samples_split': [25, 50, ...], 'n_estimators': [50, 100, ...]}"
,scoring,None
,n_jobs,-1
,refit,True
,cv,None
,verbose,1
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,n_estimators,50


In [19]:
rf = RandomForestClassifier(criterion='entropy', max_depth=6, min_samples_split=50,
                       n_estimators=50, random_state=88)
rf.fit(x_train,y_train)

,n_estimators,50
,criterion,'entropy'
,max_depth,6
,min_samples_split,50
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [20]:
root_mean_squared_error(y_test,rf.predict(x_test))

0.44048819592160315

In [21]:
print("train accuracy : ",rf.score(x_train,y_train))
print("train accuracy : ",rf.score(x_test,y_test))

train accuracy :  0.8443017656500803
train accuracy :  0.8059701492537313


In [22]:
print(classification_report(y_train,rf.predict(x_train)))
print("****************************************")
print(classification_report(y_test,rf.predict(x_test)))

              precision    recall  f1-score   support

           0       0.83      0.93      0.88       374
           1       0.87      0.72      0.79       249

    accuracy                           0.84       623
   macro avg       0.85      0.82      0.83       623
weighted avg       0.85      0.84      0.84       623

****************************************
              precision    recall  f1-score   support

           0       0.83      0.88      0.86       175
           1       0.75      0.67      0.70        93

    accuracy                           0.81       268
   macro avg       0.79      0.77      0.78       268
weighted avg       0.80      0.81      0.80       268



In [23]:
knn.fit(x_train,y_train)

,n_neighbors,5
,weights,'uniform'
,algorithm,'auto'
,leaf_size,30
,p,2
,metric,'minkowski'
,metric_params,None
,n_jobs,None


In [24]:
root_mean_squared_error(y_test,knn.predict(x_test))

0.47316022340738384

In [25]:
print("train accuracy : ",knn.score(x_train,y_train))
print("train accuracy : ",knn.score(x_test,y_test))

train accuracy :  0.8282504012841091
train accuracy :  0.7761194029850746


In [26]:
print(classification_report(y_train,knn.predict(x_train)))
print("**************************************************")
print(classification_report(y_test,knn.predict(x_test)))

              precision    recall  f1-score   support

           0       0.82      0.92      0.87       374
           1       0.85      0.69      0.76       249

    accuracy                           0.83       623
   macro avg       0.83      0.81      0.81       623
weighted avg       0.83      0.83      0.82       623

**************************************************
              precision    recall  f1-score   support

           0       0.81      0.85      0.83       175
           1       0.69      0.63      0.66        93

    accuracy                           0.78       268
   macro avg       0.75      0.74      0.75       268
weighted avg       0.77      0.78      0.77       268

